In [1]:
pip install nest_asyncio langchain nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 563.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0


In [2]:
pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.3 MB/s eta 0:00:00


In [11]:
import requests
from bs4 import BeautifulSoup
import textwrap
import time

# Function to fetch HTML content from a URL with retry mechanism
def fetch_html(url, retries=3):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
    attempt = 0
    while attempt < retries:
        try:
            response = requests.get(url, headers=headers, timeout=20)  # Increased timeout and added User-Agent
            response.raise_for_status()  # Check for request errors
            return response.content
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            attempt += 1
            if attempt < retries:
                print(f"Retrying {url} ({attempt}/{retries})...")
                time.sleep(2)  # Wait before retrying
            else:
                print(f"Failed to fetch {url} after {retries} attempts.")
                return None

# Function to remove header, footer, and extract the remaining content
def extract_main_content(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    # Remove header and footer
    if soup.header:
        soup.header.decompose()
    if soup.footer:
        soup.footer.decompose()

    # Remove page-head element by class
    page_head = soup.find(class_="page-head")
    if page_head:
        page_head.decompose()

    # Remove specific elements by their classes
    specific_classes = [
        "bUqfOz", "hEiKeJ", "gySqrp", "customHeader", "headernavbar", "breadcrumb",
        "customfooter", "content_top", "itr-season-banner", "header-wrapper",
        "common-wrapper", "common-right", "common-left", "container-fluid row",
        "bread_crumbs", "topic", "top-header", "navbar", "nav clearfix",
        "row breadcrumb-outer", "steps px-0", "menu_wrapper", "region region-user-menu",
        "myheadbtnhdr", "top-bar"
    ]

    for class_name in specific_classes:
        elements = soup.find_all(class_=class_name)
        for element in elements:
            element.decompose()

    # Get remaining text from the body
    body_content = soup.body.get_text(separator='\n').strip() if soup.body else ""
    return body_content

# Function to chunk text with overlap
def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(' '.join(chunk))
    return chunks

# List of URLs with meaningful names
urls_with_names = {
      "https://doj.gov.in/about-department/": "About Department",
    "https://doj.gov.in/memorandum-of-procedure-of-appointment-of-supreme-court-judges/": "Appointment of Supreme Court Judges",
    "https://doj.gov.in/memorandum-of-procedure-of-appointment-of-high-court-judges/": "Appointment of High Court Judges",
    "https://doj.gov.in/family-court/": "Family Court",
    "https://doj.gov.in/efiling/": "eFiling",
    "https://doj.gov.in/e-payments/": "ePayments",
    "https://doj.gov.in/fast-track-courts/": "Fast Track Courts",
    "https://doj.gov.in/ecourt-services/": "eCourt Services",
    "https://doj.gov.in/about/": "Tele-Law Services",
    "https://doj.gov.in/fast-track-special-court-ftscs/": "Fast Track Special Courts",
    "https://dashboard.doj.gov.in/scheme-for-action-research/": "Scheme for Action Research",
    "https://www.tele-law.in/": "Tele-Law Services",
    "https://www.acko.com/traffic-rules/": "Traffic Rules and Fines",
    "https://ecourtsghc.assam.gov.in/": "eCourts Services - Assam",
    "https://districts.ecourts.gov.in/app/ecourt-Motihari": "eCourt Motihari",
    "https://ecommitteesci.gov.in/mz/service/ecourts-services-mobile-app/": "eCourts Services Mobile App",
    "https://parkplus.io/c/tamilnadu-challan-information": "Tamil Nadu Challan Information",
    "https://www.bajajfinserv.in/insurance/e-challan-tamil-nadu": "E-Challan Tamil Nadu",
    "https://cleartax.in/s/e-tax-payment": "e-Tax Payment - ClearTax",
    "https://www.incometax.gov.in/iec/foportal/tax-payment-through-payment-gateway": "Tax Payment Through Payment Gateway",
    "https://www.incometax.gov.in/iec/foportal/help/e-PayTax-faqs": "e-PayTax FAQs",
    "https://lawmin.gov.in/about-us/about-the-ministry": "About Ministry of Law & Justice",
    "https://www.india.gov.in/my-government/whos-who/president": "President of India",
    "https://www.presidentofindia.gov.in/": "President of India Official Website",
    "https://www.india.gov.in/my-government/whos-who/vice-president": "Vice-President of India",
    "https://vicepresidentofindia.nic.in/": "Vice-President of India Official Website",
    "https://www.india.gov.in/my-government/whos-who/prime-minister": "Prime Minister of India",
    "https://www.pmindia.gov.in/en/pms-profile/": "Prime Minister of India Official Website",
    "https://www.india.gov.in/my-government/whos-who/council-ministers": "Council of Ministers",
    "https://www.india.gov.in/my-government/whos-who/members-parliament": "Members of Parliament",
    "https://www.pib.gov.in/PressReleasePage.aspx?PRID=2004317": "757 Fast Track Special Courts functional across the country",
    "https://ecourts.gov.in/ecourts_home/static/about-us.php": "eCourt Services",
    "https://pib.gov.in/PressReleaseIframePage.aspx?PRID=1986756": "Fast Track Schemes"
}

# Initialize an empty list for chunks
DOJ_Website_Details = []

for url, name in urls_with_names.items():
    try:
        html_content = fetch_html(url)
        if html_content:  # Proceed only if HTML content is successfully fetched
            print(f"Fetched HTML for URL: {url}")  # Debug print

            extracted_text = extract_main_content(html_content)
            if extracted_text:
                print(f"Extracted content from {url} using extract_main_content.")  # Debug print
            else:
                print(f"Can't scrape content from {url} using the <h> and <p> tags")  # Debug print
                soup = BeautifulSoup(html_content, 'html.parser')
                data = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
                scrap_data = [d.text.strip() for d in data if d.text.strip()]  # Strip and check for empty text
                extracted_text = '\n'.join(scrap_data)

            # Only chunk if there's actual text
            if extracted_text.strip():
                chunked_text = chunk_text(extracted_text, chunk_size=180, overlap=45)
                for i, chunk in enumerate(chunked_text):
                    # Wrap text to ensure it fits within the desired width
                    wrapped_text = textwrap.fill(chunk, width=80)

                    DOJ_Website_Details.append(f"{name}:\n{wrapped_text}")
            else:
                print(f"No valid content extracted from {url}.")
        else:
            print(f"Failed to fetch content from {url}")  # Debug print

    except Exception as e:
        print(f"Error fetching or processing {url}: {e}")

    time.sleep(1)




Error fetching https://doj.gov.in/about-department/: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Retrying https://doj.gov.in/about-department/ (1/3)...
Fetched HTML for URL: https://doj.gov.in/about-department/
Extracted content from https://doj.gov.in/about-department/ using extract_main_content.
Fetched HTML for URL: https://doj.gov.in/memorandum-of-procedure-of-appointment-of-supreme-court-judges/
Extracted content from https://doj.gov.in/memorandum-of-procedure-of-appointment-of-supreme-court-judges/ using extract_main_content.
Fetched HTML for URL: https://doj.gov.in/memorandum-of-procedure-of-appointment-of-high-court-judges/
Extracted content from https://doj.gov.in/memorandum-of-procedure-of-appointment-of-high-court-judges/ using extract_main_content.
Fetched HTML for URL: https://doj.gov.in/family-court/
Extracted content from https://doj.gov.in/family-court/ using extract_main_content.
Fetched HTML for URL: https://doj.gov.in/efiling/
Extra

In [12]:
# Print the chunked texts with meaningful titles
for chunk in DOJ_Website_Details:
    print(chunk)
    print("-" * 80)

About Department:
About Department Last updated: 19-04-2024 As per the Allocation of Business
(Rules), 1961, Department of Justice is a part of Ministry of Law & Justice,
Government of India. It is one of the oldest Ministries of the Government of
India. Till 31.12.2009, Department of Justice was part of Ministry of Home
Affairs and Union Home Secretary had been the Secretary of Department of
Justice. Keeping in view the increasing workload and formulating many policies
and programmes on Judicial Reforms in the country, a separate Department namely
Department of Justice was carved out from MHA and placed under the charge of
Secretary to Government of India and it started working as such from 1st
January, 2010 under the Ministry of Law & Justice. The Department is housed in
the Jaisalmer House, 26, Man Singh Road, New Delhi. The Organizational setup of
the Department includes 04 Joint Secretaries, 08 Directors/ Deputy Secretaries
and 09 Under Secretaries. The functions of the Department